# Проверка награды на испорченных презентациях

Решения 29.10 и 01.11.2026 (`docs/decisions.md`, раздел 7). Варианты и пары заморожены заранее
(`python -m presentation_rl.perturb` → `data/perturbation/variants/`), пары с конфликтом достоинств и решения
судей — `data/perturbation/conflicts/`. Здесь: сборка, награда по сетке (эмбеддер × α × λ × вес ключей),
σ по ответам T-lite (EXP-010, EXP-012), вердикты, сводка по семействам и режимам, выбор конфигурации
по правилам, зафиксированным до запуска (`perturb_eval.Rules`, `choose`), вложенная проверка по темам (4 × 3),
бутстреп по темам, чувствительность к порогам. Результат — `runs/perturbation/<дата>/`.

Два прогона (`RUN`): `grid` — EXP-014 (эмбеддер × α × λ × вес ключей при исходной награде); `ablation` — второй
прогон по решению 07.11.2026: на bge-m3, α = 0,2 исходная награда, каждое изменение по отдельности и все вместе
(`pe.VARIANTS`), для полного варианта — сетка λ × вес ключей. Фон тезисов — по 90 примерам SFT без поиска,
пороги связи визуала со слайдом — по исходникам (`tau.csv`); сводка — `ablation.csv` (Δ в абсолютных числах,
в σ своей и исходной награды, доля выполненных ожиданий по своим порогам и по старому критерию).


In [ ]:
import csv, json, os, statistics, sys, time
from dataclasses import replace
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # перед запуском проверить nvidia-smi
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from presentation_rl import perturb_eval as pe
from presentation_rl.rewards import Lemmatizer, load_key_points
from presentation_rl.tasks import load_tasks

OUT = ROOT / "runs" / "perturbation" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "runs" / "perturbation" / "embeddings"  # кеш эмбеддингов общий для всех запусков
EMBEDDERS = ["intfloat/multilingual-e5-small", "BAAI/bge-m3"]
AXES = {"alpha": [0.0, 0.2, 0.4], "length": [0.5, 1.0, 2.0], "keys": [0.5, 1.0]}  # λ — length
RUN = "ablation"  # "grid" — прогон EXP-014
ABLATION = ("BAAI/bge-m3", 0.2)  # эмбеддер и α второго прогона (решение 07.11.2026)
SFT_PLAIN = ROOT / "data" / "tasks" / "sft" / "sft_dataset.jsonl"  # чужие презентации для фона тезисов
if RUN == "ablation":
    EMBEDDERS = [ABLATION[0]]
    CURRENT = pe.Config(*ABLATION, *pe.CURRENT_WEIGHTS)  # исходная награда на bge-m3, α = 0,2
else:
    CURRENT = pe.Config(EMBEDDERS[0], 0.0, *pe.CURRENT_WEIGHTS)  # текущие коэффициенты (RewardWeights)
RUNS = [ROOT / "runs" / "variance" / "20261020_0331", ROOT / "runs" / "reward_sources" / "20261027_2150"]
M_FACTORS = [0.15, 0.25, 0.5]  # основной — 0,25 (pe.M_SHARE); остальные — чувствительность
RULES = pe.Rules()
CODEX_KINDS = ["repeat_vs_missing"]  # где Codex дополняет автора
KEYS_MIN = 10  # меньше решённых пар про ключи — вес ключей не выбирается
print(OUT)

In [ ]:
def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        return
    fields = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="", encoding="utf-8-sig") as f:  # utf-8-sig — для Excel
        writer = csv.DictWriter(f, fields)
        writer.writeheader()
        writer.writerows(rows)


points = load_key_points(ROOT / "data" / "tasks" / "key_points" / "key_points.json")
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json")}
items, pairs = pe.load_variants(ROOT / "data" / "perturbation" / "variants")
contexts = {key[0]: item.context for key, item in items.items() if key[1] == "original"}
MODES = {t: ("search" if c["search"] else "plain") for t, c in contexts.items()}
natural = pe.load_natural(RUNS, tasks, set(contexts))
conflict_items, conflicts = pe.load_conflicts(ROOT / "data" / "perturbation" / "conflicts", contexts)
all_items = {**items, **natural, **conflict_items}
print(f"вариантов {len(items)}, пар {len(pairs)}, ответов T-lite {len(natural)}, презентаций из пар с конфликтом "
      f"{len(conflict_items)}")
start = time.time()
pe.build_all(all_items)
pairs, dropped = pe.check_pairs(pairs, items)
print(f"сборка {time.time() - start:.0f} с; пар после предусловий {len(pairs)}, отброшено {len(dropped)}")
for line in dropped:
    print("  -", line)

## Судья для пар с конфликтом

Решение 01.11.2026 (`decisions.md`, раздел 7): судья — автор (50 пар, разметка вслепую); на парах «повторы против
отсутствия заключения», которые автор не размечал, — Codex (с автором на этом виде совпал в 8 из 10). Ответы
«равны» и пары, где Codex в двух порядках выбрал разное, не учитываются. Вес ключей выбирается по парам
с конфликтом, только если решённых пар про ключи не меньше 10; иначе — текущий (0,5).


In [ ]:
pe.combine_judges(conflicts, "author", "codex", CODEX_KINDS, "judge")
JUDGE = "judge"
for kind in sorted({c["kind"] for c in conflicts}):
    part = [c for c in conflicts if c["kind"] == kind]
    print(f"  {kind}: решено автором {sum(c['judges'].get('author') not in (None, 'equal') for c in part)}, "
          f"сводным судьёй {sum(c['judges'].get('judge') not in (None, 'equal') for c in part)} из {len(part)}; "
          f"Codex определился в {sum(c['judges'].get('codex') is not None for c in part)}")
same, both = pe.judges_agreement(conflicts, "codex", "author")
print(f"согласие Codex и автора (второе мнение, для отчёта): {same} из {both}")
decided = {d: sum(c["judges"].get(JUDGE) not in (None, "equal") for c in conflicts if c["decides"] == d)
           for d in ("length", "keys")}
CHOOSE_KEYS = decided["keys"] >= KEYS_MIN
print("решённых пар:", decided, "| вес ключей", "выбирается" if CHOOSE_KEYS else "остаётся текущим (мало пар)")


In [ ]:
if RUN == "grid":
    configs = pe.grid(EMBEDDERS, AXES["alpha"], AXES["length"], AXES["keys"])
else:
    configs = pe.ablation_configs(*ABLATION, AXES["length"], AXES["keys"])
foreign = pe.load_foreign(SFT_PLAIN) if RUN == "ablation" else None
originals = {key[0]: item.deck for key, item in items.items() if key[1] == "original"}
domains = {t: tasks[t].request.domain for t in originals}
TAU = {}  # пороги визуала по эмбеддерам (нужны и для новых исходников)


def run_grid(configs):
    lemmatizer = Lemmatizer()
    scores, sigmas = {}, {}
    for name in sorted({c.embedder for c in configs}):
        start = time.time()
        embedder = pe.CachedEmbedder(name, CACHE)
        print(name, embedder.signature, "в кеше", len(embedder.vectors))
        tau = None
        if any(c.variant != "base" for c in configs if c.embedder == name):
            tau, tau_report = pe.calibrate_tau(originals, domains, embedder)
            TAU[name] = tau
            write_csv(OUT / f"tau_{name.split('/')[-1]}.csv", tau_report)
            print("  пороги визуала:", tau, "| без порога (половина бонуса):",
                  [r["kind"] for r in tau_report if not r["verified"]])
        for config in [c for c in configs if c.embedder == name]:
            scores[config] = pe.score_items(all_items, points, config, embedder, lemmatizer, foreign, tau)
            sigmas[config], _ = pe.sigma(scores[config])
            embedder.save()
            print(f"  {config.name}: σ = {sigmas[config]:.4f}")
        print(f"  {time.time() - start:.0f} с")
    return scores, sigmas


scores, sigmas = run_grid(configs)


def evaluate(scores, sigmas, m_factor=pe.M_SHARE):
    rows_by, conf_by = {}, {}
    for config, s in scores.items():
        m, eps = m_factor * sigmas[config], pe.EPS_SHARE * sigmas[config]
        rows_by[config] = pe.evaluate_pairs(pairs, s, m, eps)
        conf_by[config] = pe.conflict_rows(conflicts, s, JUDGE, eps) if JUDGE else []
    return rows_by, conf_by


rows_by, conf_by = evaluate(scores, sigmas)
write_csv(OUT / "sigma.csv", ({"config": c.name, **g} for c, s in scores.items() for g in pe.sigma(s)[1]))
write_csv(OUT / "components.csv", ({"config": c.name, "key": "/".join(map(str, k)), **v}
                                   for c, s in scores.items() for k, v in s.items()))
write_csv(OUT / "pairs.csv", ({"config": c.name, **r} for c, rows in rows_by.items() for r in rows))

## Величина при текущих коэффициентах

Сдвиг награды и целевого компонента по видам искажений в сравнении с естественным разбросом σ: штраф должен быть
заметен на фоне σ и не перевешивать содержание.

In [ ]:
summary = []
for config, rows in rows_by.items():
    for entry in pe.summarize(rows):
        entry["d_total_median_sigma"] = entry["d_total_median"] / sigmas[config]
        summary.append({"config": config.name, "sigma": sigmas[config], **entry})
write_csv(OUT / "summary.csv", summary)
cols = ["family", "kind", "mode", "n", "d_total_median", "d_total_median_sigma", "d_target_median", "total_ok",
        "total_violation", "component_ok", "component_down_total_up"]
print(f"текущие коэффициенты: {CURRENT.name}, σ = {sigmas[CURRENT]:.4f}")
for e in (s for s in summary if s["config"] == CURRENT.name):
    print("  ".join(f"{e[c]:.3f}" if isinstance(e[c], float) else str(e[c]) for c in cols))
print("провал в одном режиме (предупреждения, без отсева):", pe.mode_failures(rows_by[CURRENT], RULES))

## Абляции (второй прогон)

Каждое изменение награды по отдельности и все вместе при текущих λ и весе ключей: медиана Δ награды по видам
искажений в σ исходной награды и доля выполненных ожиданий по старому критерию (пороги от σ исходной награды).
Это регрессия на тех же парах, по которым изменения придуманы; независимая проверка — на новых исходниках.


In [ ]:
if RUN == "ablation":
    ablation = pe.ablation_rows(pairs, scores, sigmas, CURRENT)
    write_csv(OUT / "ablation.csv", ablation)
    shown = [c for c in configs if (c.length, c.keys) == pe.CURRENT_WEIGHTS]
    print("σ:", "  ".join(f"{c.variant}={sigmas[c]:.4f}" for c in shown))
    table = {}
    for r in ablation:
        table.setdefault((r["family"], r["kind"], r["mode"]), {})[r["variant"], r["config"]] = r
    names = [c.variant for c in shown]
    print(f"{'вид искажения':42s}" + "".join(f"{n[:9]:>10s}" for n in names) + "   (медиана Δ в σ исходной / доля ok по старому)")
    for key, row in sorted(table.items()):
        cells = [row.get((c.variant, c.name)) for c in shown]
        print(f"{'/'.join(map(str, key))[:42]:42s}" + "".join(
            f"{r['d_sigma_base']:>+6.2f}/{r['total_ok_base_criterion'] or 0:.1f}"[:10].rjust(10) if r else " " * 10
            for r in cells))


## Выбор конфигурации

In [ ]:
def select_all(rows_by, conf_by, rules=RULES):
    allowed = [c for c in rows_by if CHOOSE_KEYS or c.keys == pe.CURRENT_WEIGHTS[1]]
    tallies = {c: pe.tally(rows_by[c]) for c in allowed}
    ctallies = {c: pe.conflict_tally(conf_by[c]) for c in allowed} if JUDGE else None
    return pe.nested_cv(tallies, ctallies, MODES, rules=rules), tallies


cv, tallies = select_all(rows_by, conf_by)
for fold in cv["outer"]:
    print(fold)
print("итог:", cv["final"], cv["final_info"])
print(f"устойчив: {cv['stable']}, оптимизм: {cv['optimism']}, надёжен: {cv['reliable']}")
final = next(c for c in configs if c.name == cv["final"])
chosen_axes = ["alpha"] + (["length"] + (["keys"] if CHOOSE_KEYS else []) if JUDGE and "kept_current" not in cv["final_info"] else [])
extend = pe.edge(final, configs, chosen_axes)
print("на краю сетки — расширить:", extend or "нет")
if JUDGE:
    print("различают ли пары с конфликтом λ и вес ключей:", pe.identifiable(conf_by))
ci = pe.bootstrap(tallies[final], sorted(MODES))
shares = pe.family_shares(tallies[final], {t: 1 for t in MODES})
write_csv(OUT / "bootstrap.csv", [{"family": f, "share": shares.get(f), "low": lo, "median": med, "high": hi}
                                  for f, (lo, med, hi) in sorted(ci.items())])
(OUT / "cv.json").write_text(json.dumps(cv, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
for f, (lo, med, hi) in sorted(ci.items()):
    print(f"{f:14s} {shares.get(f, float('nan')):.2f}  [{lo:.2f}; {hi:.2f}]")

## Расширение сетки (одно, по правилу)

Если выбранная конфигурация на краю сетки по α, λ или весу ключей — добавить значение за краем и повторить
выбор теми же правилами. Расширение делается один раз.

In [ ]:
if extend and RUN == "grid":
    for axis, value in extend.items():
        AXES[axis] = sorted(AXES[axis] + [value])
    extra = [c for c in pe.grid(EMBEDDERS, AXES["alpha"], AXES["length"], AXES["keys"]) if c not in scores]
    more_scores, more_sigmas = run_grid(extra)
    scores.update(more_scores)
    sigmas.update(more_sigmas)
    configs = list(scores)
    rows_by, conf_by = evaluate(scores, sigmas)
    cv, tallies = select_all(rows_by, conf_by)
    final = next(c for c in configs if c.name == cv["final"])
    print("после расширения:", cv["final"], "| устойчив:", cv["stable"], "| надёжен:", cv["reliable"])
    (OUT / "cv_extended.json").write_text(json.dumps(cv, ensure_ascii=False, indent=1, default=str), encoding="utf-8")

## Чувствительность к порогам (для статьи)

Выбор делается только по основным правилам; здесь — меняется ли победитель при других порогах.

In [ ]:
variants = []
for m_factor in M_FACTORS:
    rows_m, conf_m = evaluate(scores, sigmas, m_factor)
    for fail_core, fail_other in ((0.6, 0.4), (0.7, 0.5), (0.8, 0.6)):
        for close_scale in (0.5, 1.0, 2.0):
            rules = replace(RULES, fail_core=fail_core, fail_other=fail_other, close_scale=close_scale)
            cv_s, _ = select_all(rows_m, conf_m, rules)
            variants.append({"m": m_factor, "fail_core": fail_core, "fail_other": fail_other, "close": close_scale,
                             "chosen": cv_s["final"], "same_as_main": cv_s["final"] == final.name,
                             "feasible": cv_s["final_info"]["feasible"], "exempt": ";".join(cv_s["final_info"]["exempt"]),
                             "stable": cv_s["stable"], "optimism": cv_s["optimism"]})
write_csv(OUT / "sensitivity.csv", variants)
print(f"победитель тот же в {sum(v['same_as_main'] for v in variants)} из {len(variants)} вариантов порогов")

## Независимая проверка на новых исходниках

Решение 07.11.2026: изменения награды придуманы по парам 16 исходников, поэтому их проверка на тех же парах — регрессия.
Независимая проверка — на новых исходниках тем validation (`data/perturbation/holdout`, `python -m presentation_rl.perturb
--base data/perturbation/holdout`): те же конфигурации, пороги m и ε — от σ основного набора, по каждой — доля
выполненных ожиданий по семействам. Результат — `holdout_ablation.csv`, `holdout_pairs.csv`.


In [ ]:
HOLDOUT = ROOT / "data" / "perturbation" / "holdout" / "variants"
if RUN == "ablation" and HOLDOUT.exists():
    hold_items, hold_pairs = pe.load_variants(HOLDOUT)
    pe.build_all(hold_items)
    hold_pairs, hold_dropped = pe.check_pairs(hold_pairs, hold_items)
    print(f"новые исходники: вариантов {len(hold_items)}, пар {len(hold_pairs)}, отброшено {len(hold_dropped)}")
    lemmatizer = Lemmatizer()
    embedder = pe.CachedEmbedder(ABLATION[0], CACHE)
    hold_scores = {c: pe.score_items(hold_items, points, c, embedder, lemmatizer, foreign, TAU.get(c.embedder))
                   for c in configs}
    embedder.save()
    write_csv(OUT / "holdout_ablation.csv", pe.ablation_rows(hold_pairs, hold_scores, sigmas, CURRENT))
    hold_rows = {c: pe.evaluate_pairs(hold_pairs, s, pe.M_SHARE * sigmas[c], pe.EPS_SHARE * sigmas[c])
                 for c, s in hold_scores.items()}
    write_csv(OUT / "holdout_pairs.csv", ({"config": c.name, **r} for c, rows in hold_rows.items() for r in rows))
    hold_weights = {t: 1 for t in {p["task"] for p in hold_pairs}}
    for c in configs:
        shares = pe.family_shares(pe.tally(hold_rows[c]), hold_weights)
        print(f"{c.name:45s} " + "  ".join(f"{f}={v:.2f}" for f, v in sorted(shares.items()))
              + ("  ← выбрана" if c == final else ""))
